# E014 agentic replay benchmark: sglang on 4xL4 (offline)

Generated by `kaggle/build_notebook.py --variant replay` from the team repo (`kaggle/src/replay_bench.py`, `kaggle/src/solver.py`). Replays recorded E012 tool-use transcripts turn by turn and measures throughput, prefix-cache hit rate and evictions at several concurrency levels. Internet off; time-boxed to 58 min from the first cell.

In [ ]:
import json, os, time
os.environ['ARC_T0'] = str(time.time())
os.environ['ARC_MODE'] = 'bench'
SETTINGS = {
 "ARC_SERVER": "sglang",
 "ARC_BENCH_STOP_MIN": "58",
 "ARC_REPLAY_PLAN": "[{\"name\": \"sgl_radix\", \"max_seqs\": 36, \"env\": {\"ARC_SGL_MAMBA_STRATEGY\": \"extra_buffer_lazy\", \"ARC_SGL_SKIP_LOCK\": \"1\", \"ARC_SGL_MAMBA_SLOTS\": \"108\"}, \"fallback_envs\": [{\"ARC_SGL_SKIP_LOCK\": \"1\", \"ARC_SGL_MAMBA_SLOTS\": \"144\"}, {}], \"levels\": [[16, 300], [24, 300], [35, 300], [48, 300]]}, {\"name\": \"sgl_radix_hicache\", \"max_seqs\": 36, \"base\": \"sgl_radix\", \"extra_env\": {\"ARC_SGL_HICACHE\": \"2\"}, \"levels\": [[24, 300], [35, 300], [48, 300]]}]"
}
for k, v in SETTINGS.items():
    os.environ[k] = v if isinstance(v, str) else json.dumps(v)
os.makedirs('/kaggle/working/arc_solver', exist_ok=True)
print({k: v for k, v in os.environ.items() if k.startswith('ARC_')})

In [ ]:
%%writefile /kaggle/working/arc_solver/solver.py
"""ARC Prize 2026 (ARC-AGI-2) offline solver: Qwen3.8-27B-FP8 on vLLM (TP=4, 4xL4), no tools.

One self-contained file; kaggle/build_notebook.py embeds it in the Kaggle notebook. Modes (env ARC_MODE):

- submit:   full run on the competition test file (used automatically on the competition rerun).
- fallback: write a valid fallback submission.json and exit (used for the commit run before submitting).
- bench:    test run: provenance hashes, server start, long-context decode benchmark, short end-to-end run on
            evaluation tasks with a near deadline, scoring against the evaluation solutions.
            ARC_HASH=0 / ARC_BENCH_DECODE=0 skip the first two (validation run: ARC_E2E_TASKS, ARC_E2E_MIN).

Robustness rules: a complete, valid submission.json is written before anything else and rewritten atomically
after every finished attempt; every failure path keeps the last valid file.

Prompt, parser and Qwen raw-template rendering follow evals/arc_harness/arcprize.py and evals/run_qwen_generate.py
(ARC Prize official prompt, tier xhigh, temp 1.0, top_p 0.95, top_k 20, forced final answer at the budget).
"""
import asyncio
import glob
import hashlib
import json
import os
import re
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor

T0 = float(os.environ.get("ARC_T0", time.time()))  # notebook start (set by the first cell)
MODE = os.environ.get("ARC_MODE", "submit")
WORK = "/kaggle/working"
WORK = os.environ.get("ARC_WORK", WORK)
OUT_SUB = os.path.join(WORK, "submission.json")
LOG_DIR = os.path.join(WORK, "arc_logs")
PORT = 8000
BASE = f"http://127.0.0.1:{PORT}"
SERVED = os.environ.get("ARC_MODEL_ID", "qwen3.8-27b")
# Off-Kaggle validation: use an already running OpenAI-compatible server and skip install / launch.
EXTERNAL = os.environ.get("ARC_EXTERNAL_SERVER")  # e.g. http://localhost:30000/v1
if EXTERNAL:
    BASE = EXTERNAL.rstrip("/").removesuffix("/v1")
INPUT = os.environ.get("ARC_INPUT", "/kaggle/input")

# ---------------------------------------------------------------- settings (env overridable)

def envf(name, default):
    return type(default)(os.environ.get(name, default))

FORCE_AT_S = envf("ARC_FORCE_AT_S", 11.0 * 3600)    # force final answers for all in-flight attempts
HARD_STOP_S = envf("ARC_HARD_STOP_S", 11.5 * 3600)  # stop everything, final write (notebook guard: 11.75 h)
B_MAX1 = envf("ARC_B_MAX1", 96000)                  # attempt-1 budget cap (thinking + answer)
B_MIN1 = envf("ARC_B_MIN1", 16000)
B_MAX2 = envf("ARC_B_MAX2", 64000)
B_MIN2 = envf("ARC_B_MIN2", 32000)                  # attempt 2 is launched only if it can get this much
FINAL_TOKENS = envf("ARC_FINAL_TOKENS", 3000)
RATE_PRIOR = envf("ARC_RATE_PRIOR", 350.0)          # tok/s before the run measures its own rate
MAX_MODEL_LEN = envf("ARC_MAX_MODEL_LEN", 131072)
MAX_SEQS = envf("ARC_MAX_SEQS", 64)
CONCURRENCY = envf("ARC_CONCURRENCY", 0)            # 0 = from KV capacity
TOKENS_PER_SEQ_KV = envf("ARC_KV_PER_SEQ", 30000)   # KV tokens assumed per sequence when sizing concurrency
MTP = envf("ARC_MTP", 0)                            # speculative tokens (0 = off)
TIER = os.environ.get("ARC_TIER", "xhigh")
TEMPERATURE = envf("ARC_TEMPERATURE", 1.0)
GPU_UTIL = envf("ARC_GPU_UTIL", 0.95)
SERVER = os.environ.get("ARC_SERVER", "vllm")        # vllm (user's vllm-0.23.0 cu130 source) | sglang (ours)
AGENT = os.environ.get("ARC_AGENT", "tools")         # tools (E012 agent_qwen) | plain (v1 no-tool prompt)
TOOLS_CAP = envf("ARC_TOOLS_CAP", 48000)             # E012: hard cap per attempt (min with the adaptive share)
TOOLS_SOFT = envf("ARC_TOOLS_SOFT", 24000)
TOOLS_TIER = os.environ.get("ARC_TOOLS_TIER", "medium")
TOOL_TIMEOUT = envf("ARC_TOOL_TIMEOUT", 30.0)
LOCAL_FORCE_MIN = envf("ARC_LOCAL_FORCE_MIN", 40.0)  # local validation: force at max(T0 + this, ready + 20 min)
LOCAL_FORCE_CAP_MIN = envf("ARC_LOCAL_FORCE_CAP_MIN", 60.0)  # ... but never later than T0 + this
LOCAL_READY_MIN = envf("ARC_LOCAL_READY_MIN", 0.0)  # > 0: local force = server ready + this (overrides the above)
LOCAL_HARD_MIN = envf("ARC_LOCAL_HARD_MIN", 8.0)    # local hard stop = force + this
LOCAL_TASKS = os.environ.get("ARC_LOCAL_TASKS", "")  # comma-separated subset of the local test set
PUBLIC_TEST_IDS_SHA256 = "4cab35dea7bde6ee39bbfcfdcfc101153e19f333c14b68c68b84a7a3278c265e"  # public test file
PHASES = {}  # phase timings (s since T0), written to the summary
PKG = os.path.join(WORK, "arc_solver")  # evals/arc_harness + notes/prompts + solver.py, written by the notebook

PROMPT_TEMPLATE = """You are participating in a puzzle solving competition. You are an expert at solving puzzles.

Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.

Respond in the format of the training output examples

--Training Examples--
{training_examples}
--End of Training Examples--

--Test Input--
{test_input}
--End of Test Input--

Your response:"""  # noqa: F821  (ARC Prize official prompt, injected by build_notebook.py)

CLOSE = "</think>"
FORCE_TEXT = "\n\nI have to give my final answer now.\n</think>\n\n"
LOW_YIELD = 20
TIER_SYSTEM = {
    "xhigh": "Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, "
             "consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final "
             "answer.",
    "medium": None,
}


def log(*a):
    el = time.time() - T0
    print(f"[{el / 3600:5.2f}h {time.strftime('%H:%M:%S')}]", *a, flush=True)


# ---------------------------------------------------------------- prompt / parse (verbatim from arc_harness)

def build_prompt(train, test_input):
    ex = ""
    for i, pair in enumerate(train):
        ex += f"--Example {i}-- \n\n INPUT: \n\n" + json.dumps(pair["input"]) + "\n\n"
        ex += "OUTPUT: \n\n" + json.dumps(pair["output"]) + "\n\n"
    return PROMPT_TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))


def render_prompt(user_content, tier=TIER):
    s = ""
    if TIER_SYSTEM.get(tier):
        s += f"<|im_start|>system\n{TIER_SYSTEM[tier]}<|im_end|>\n"
    return s + f"<|im_start|>user\n{user_content.strip()}<|im_end|>\n<|im_start|>assistant\n<think>\n"


def answer_section(gen):
    i = gen.rfind(CLOSE)
    return None if i < 0 else gen[i + len(CLOSE):]


def is_grid(x):
    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)


def extract_from_boxed(s):
    m = re.search(r"\\boxed\{(.*?)\}", s, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            return g if is_grid(g) else None
        except json.JSONDecodeError:
            return None
    return None


def backscan_json_parser(s):
    end = max(s.rfind("]"), s.rfind("}"))
    if end < 0:
        return None
    close = s[end]
    open_ = "[" if close == "]" else "{"
    depth = 1
    for i in range(end - 1, -1, -1):
        depth += (s[i] == close) - (s[i] == open_)
        if depth == 0:
            try:
                g = json.loads(s[i:end + 1])
                return g if is_grid(g) else None
            except json.JSONDecodeError:
                return None
    return None


def parse(s):
    return extract_from_boxed(s or "") or backscan_json_parser(s or "")


def valid_grid(grid):
    return (isinstance(grid, list) and 1 <= len(grid) <= 30
            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)
            and len({len(row) for row in grid}) == 1
            and all(type(v) is int and 0 <= v <= 9 for row in grid for v in row))


# ---------------------------------------------------------------- paths

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None


def comp_file(name):
    return (find_one(f"{INPUT}/competitions/arc-prize-2026-arc-agi-2/{name}")
            or find_one(f"{INPUT}/arc-prize-2026-arc-agi-2/{name}")
            or find_one(f"{INPUT}/**/{name}"))


def model_dir():
    for pat in [f"{INPUT}/models/mikedan7/qwen3-8-27b-fp8-official/**/config.json",
                f"{INPUT}/qwen3-8-27b-fp8-official/**/config.json",
                f"{INPUT}/**/qwen3-8-27b-fp8*/**/config.json"]:
        p = find_one(pat)
        if p:
            return os.path.dirname(p)
    raise FileNotFoundError("model dir")


def wheel_dir():
    """Our own wheel sources (attached as kernel sources): SGLang 0.5.20 cu130 or the user's vLLM 0.23.0 cu128."""
    pat = "sglang-0.5.20-*.whl" if SERVER == "sglang" else "vllm-0.23.0*.whl"
    p = find_one(f"{INPUT}/notebooks/chankhavu/**/{pat}") or find_one(f"{INPUT}/**/{pat}")
    if not p:
        raise FileNotFoundError(f"wheel source for {SERVER}")
    return os.path.dirname(p)


# ---------------------------------------------------------------- submission

class Submission:
    def __init__(self, tasks):
        self.tasks = tasks
        self.answers = {}  # (tid, pair) -> {attempt_index: grid}

    def fallback(self, tid, p):
        g = self.tasks[tid]["test"][p]["input"]
        return g if valid_grid(g) else [[0]]

    def build(self):
        sub = {}
        for tid, task in self.tasks.items():
            lst = []
            for p in range(len(task["test"])):
                got = [g for _, g in sorted(self.answers.get((tid, p), {}).items()) if valid_grid(g)]
                a1 = got[0] if got else self.fallback(tid, p)
                a2 = next((g for g in got[1:] if g != a1), None)
                if a2 is None:
                    a2 = self.fallback(tid, p) if a1 != self.fallback(tid, p) else a1
                lst.append({"attempt_1": a1, "attempt_2": a2})
            sub[tid] = lst
        return sub

    def write(self):
        sub = self.build()
        tmp = OUT_SUB + ".tmp"
        with open(tmp, "w") as f:
            json.dump(sub, f)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, OUT_SUB)


def check_submission(path, tasks):
    sub = json.load(open(path))
    assert set(sub) == set(tasks), "task ids differ"
    for tid, task in tasks.items():
        assert len(sub[tid]) == len(task["test"]), tid
        for slot in sub[tid]:
            assert valid_grid(slot["attempt_1"]) and valid_grid(slot["attempt_2"]), tid
    return True


# ---------------------------------------------------------------- environment

def sh(cmd, timeout=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout or "") + (r.stderr or "")


def driver_env():
    """Host driver libcuda first (images also carry an older compat libcuda); set before any CUDA process starts."""
    real = "/usr/local/nvidia/lib64"
    if os.path.exists(real + "/libcuda.so"):
        os.environ["LD_LIBRARY_PATH"] = real + ":" + os.environ.get("LD_LIBRARY_PATH", "")
        os.environ["LD_PRELOAD"] = real + "/libcuda.so"


def cuda_toolkit_env(env=None, shim=None):
    """CUDA 13 toolkit from the pip wheels for JIT builds (SGLang's sglang-kernel JIT, FlashInfer, tvm-ffi).

    The image has no /usr/local/cuda/bin/nvcc; our SGLang wheel source installs nvidia-cuda-nvcc/crt/nvvm/cccl/runtime
    into site-packages/nvidia/cu13. JIT builders expect $CUDA_HOME/{bin/nvcc,include,nvvm} and link with
    `-L$CUDA_HOME/lib64 -lcudart`, while the wheels ship lib/ with only libcudart.so.13, so a shim directory of
    symlinks gives the expected layout. Updates `env` (default os.environ); returns the shim path or None."""
    env = os.environ if env is None else env
    shim = shim or os.environ.get("ARC_CUDA_SHIM", "/tmp/cuda13")
    import site
    roots = list(dict.fromkeys(site.getsitepackages() + glob.glob("/usr/local/lib/python3*/dist-packages")
                               + glob.glob("/usr/lib/python3/dist-packages")))
    cu = next((os.path.join(r, "nvidia/cu13") for r in roots if os.path.exists(os.path.join(r, "nvidia/cu13/bin/nvcc"))),
              None)
    if cu is None:
        log("cuda toolkit: no nvidia/cu13/bin/nvcc in", roots)
        return None
    os.makedirs(os.path.join(shim, "lib64"), exist_ok=True)
    for d in ("bin", "include", "nvvm", "lib", "share"):
        src, dst = os.path.join(cu, d), os.path.join(shim, d)
        if os.path.exists(src) and not os.path.lexists(dst):
            os.symlink(src, dst)
    for f in os.listdir(os.path.join(cu, "lib")):
        dst = os.path.join(shim, "lib64", f)
        if not os.path.lexists(dst):
            os.symlink(os.path.join(cu, "lib", f), dst)
    for name in ("cudart", "nvrtc"):
        real = sorted(glob.glob(os.path.join(cu, "lib", f"lib{name}.so.*")))
        dst = os.path.join(shim, "lib64", f"lib{name}.so")
        if real and not os.path.lexists(dst):
            os.symlink(real[0], dst)
    env.update(CUDA_HOME=shim, CUDA_PATH=shim, CUDA_ROOT=shim,
               PATH=os.path.join(shim, "bin") + ":" + env.get("PATH", ""),
               LD_LIBRARY_PATH=(env.get("LD_LIBRARY_PATH", "") + ":" + os.path.join(shim, "lib64")).lstrip(":"),
               TORCH_CUDA_ARCH_LIST=os.environ.get("ARC_CUDA_ARCH", "8.9"))
    return shim


def sglang_jit_env(env):
    """SGLang JIT kernel cache: a writable dir, seeded from a pre-built cache attached as input if there is one."""
    import shutil
    jit = os.environ.get("ARC_SGL_JIT_DIR", "/tmp/sglang_jit")
    seed = find_one(f"{INPUT}/**/sglang_jit_cache/sm89") if os.path.isdir(INPUT) else None
    if seed and not os.path.exists(os.path.join(jit, "sm89")):
        shutil.copytree(seed, os.path.join(jit, "sm89"), symlinks=True)
        log("sglang JIT cache seeded from", seed)
    os.makedirs(jit, exist_ok=True)
    env["SGLANG_JIT_CACHE_DIR"] = jit
    return jit


def install_server():
    wd = wheel_dir()
    t = time.time()
    if SERVER == "sglang":
        cmd = f"{sys.executable} -m pip install --no-index --find-links {wd} -r {wd}/requirements.lock"
    else:
        cmd = (f"{sys.executable} -m pip install --no-index --find-links {wd} vllm==0.23.0 flashinfer-python "
               f"flashinfer-jit-cache")
    code, out = sh("uv " + cmd.replace(f"{sys.executable} -m pip install", "pip install --system --python "
                                       + sys.executable), timeout=2400)
    log(f"uv offline install of {SERVER} from {wd}: rc={code} in {time.time() - t:.0f}s")
    if code != 0:
        print(out[-1500:])
        t = time.time()
        code, out = sh(cmd, timeout=2400)
        log(f"pip offline install of {SERVER} from {wd}: rc={code} in {time.time() - t:.0f}s")
    if code != 0:
        print(out[-4000:])
        raise RuntimeError("pip install failed")
    mod = "sglang" if SERVER == "sglang" else "vllm"
    code, out = sh(f"{sys.executable} -c 'import torch; print(torch.version.cuda)'", timeout=600)
    if out.strip().splitlines()[-1:] == ["None"]:
        # A CPU torch of the same version in the image satisfies the pin and is kept: force the CUDA build.
        tw = sorted(glob.glob(f"{wd}/torch-*.whl")) + sorted(glob.glob(f"{wd}/torchvision-*.whl")) + \
            sorted(glob.glob(f"{wd}/torchaudio-*.whl"))
        code, out2 = sh(f"{sys.executable} -m pip install --no-index --force-reinstall --no-deps " + " ".join(tw),
                        timeout=1800)
        log(f"CPU torch found; reinstalled CUDA torch from the wheel source: rc={code}")
    code, out = sh(f"{sys.executable} -c 'import torch, {mod}; print(torch.__version__, torch.version.cuda, "
                   f"torch.cuda.is_available(), torch.cuda.device_count(), {mod}.__version__)'", timeout=600)
    log("torch/server:", out.strip().splitlines()[-1] if out.strip() else out)
    if SERVER == "sglang":
        shim = cuda_toolkit_env()
        code, out = sh(f"{shim}/bin/nvcc --version | tail -2", timeout=120) if shim else (1, "no shim")
        log(f"cuda toolkit shim {shim}: nvcc rc={code} {out.strip().splitlines()[-1:] if out.strip() else ''}")
    if SERVER == "vllm":
        _, ldd = sh(f"ldd $({sys.executable} -c 'import glob, os, vllm; "
                    f"print(glob.glob(os.path.dirname(vllm.__file__) + \"/_C*.so\")[0])') | grep -i 'not found'",
                    timeout=120)
        log("vllm._C missing libraries:", ldd.strip() or "none")
    _, mem = sh("free -g | head -2", timeout=30)
    log("host memory:", " | ".join(mem.strip().splitlines()))
    _, freeze = sh(f"{sys.executable} -m pip freeze", timeout=300)
    open(os.path.join(LOG_DIR, "pip_freeze.txt"), "w").write(freeze)
    log("installed:", " ".join(l for l in freeze.splitlines() if l.split("==")[0].lower() in
                               ("vllm", "torch", "transformers", "flashinfer-python", "triton", "sglang", "numpy")))


def file_hashes(paths, workers=16):
    def h(p):
        s256, s1 = hashlib.sha256(), hashlib.sha1()
        size = os.path.getsize(p)
        s1.update(f"blob {size}\0".encode())
        with open(p, "rb") as f:
            while True:
                b = f.read(1 << 24)
                if not b:
                    break
                s256.update(b)
                s1.update(b)
        return p, {"size": size, "sha256": s256.hexdigest(), "git_sha1": s1.hexdigest()}
    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(h, paths))


def preload(mdir, workers=16):
    files = [os.path.join(r, f) for r, _, fs in os.walk(mdir) for f in fs]

    def rd(p):
        with open(p, "rb") as f:
            while f.read(1 << 26):
                pass
    t = time.time()
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(rd, files))
    log(f"preloaded {len(files)} model files in {time.time() - t:.0f}s")


def launch_server(mdir):
    env = os.environ.copy()
    env.update(PYTORCH_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1",
               TRANSFORMERS_OFFLINE="1", OMP_NUM_THREADS="1")
    if SERVER == "sglang":
        cmd = [sys.executable, "-m", "sglang.launch_server", "--model-path", mdir, "--served-model-name", SERVED,
               "--host", "127.0.0.1", "--port", str(PORT), "--tp-size", "4", "--kv-cache-dtype", "fp8_e4m3",
               "--context-length", str(MAX_MODEL_LEN), "--mem-fraction-static", os.environ.get("ARC_SGL_MEM", "0.85"),
               "--max-running-requests", str(MAX_SEQS), "--trust-remote-code",
               "--attention-backend", os.environ.get("ARC_SGL_ATTN", "triton"), "--log-level", "info",
               "--enable-cache-report"]
        # Radix (prefix) cache is mandatory for the tool agent (every turn re-sends the whole transcript); the
        # hybrid model's Mamba state slots bound the running batch: max_running = slots / slots-per-request
        # (5 with extra_buffer + overlap; 3 with extra_buffer_lazy + SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK=1).
        if os.environ.get("ARC_RADIX", "1") != "1":
            cmd.append("--disable-radix-cache")
        if os.environ.get("ARC_SGL_MAMBA_SLOTS"):
            cmd += ["--max-mamba-cache-size", os.environ["ARC_SGL_MAMBA_SLOTS"]]
        if os.environ.get("ARC_SGL_MAMBA_STRATEGY"):
            cmd += ["--mamba-radix-cache-strategy", os.environ["ARC_SGL_MAMBA_STRATEGY"]]
        if os.environ.get("ARC_SGL_SKIP_LOCK", "0") == "1":
            env["SGLANG_OPT_MAMBA_SKIP_DECODE_LOCK"] = "1"
        if envf("ARC_SGL_HICACHE", 0) > 0:  # host-memory KV offload (hierarchical cache), ratio = host / device
            cmd += ["--enable-hierarchical-cache", "--hicache-ratio", os.environ["ARC_SGL_HICACHE"],
                    "--hicache-write-policy", os.environ.get("ARC_SGL_HICACHE_POLICY", "write_through")]
        cuda_toolkit_env(env)
        sglang_jit_env(env)
    else:
        attn = "TRITON_ATTN"  # FlashInfer fp8-KV kernels need SM90+; L4 is SM89
        cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", mdir,
               "--served-model-name", SERVED, "--host", "127.0.0.1", "--port", str(PORT),
               "--tensor-parallel-size", "4", "--gpu-memory-utilization", str(GPU_UTIL),
               "--max-model-len", str(MAX_MODEL_LEN), "--max-num-seqs", str(MAX_SEQS),
               "--kv-cache-dtype", "fp8", "--attention-backend", attn, "--enable-prefix-caching",
               "--trust-remote-code", "--no-async-scheduling"]
        if MTP > 0:
            cmd += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": MTP})]
        env.update(VLLM_FORCE_ATTN_BACKEND=attn, VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS="1")
    cmd += json.loads(os.environ.get("ARC_SERVER_EXTRA", "[]"))  # extra server flags (bench / experiments)
    log("server cmd:", " ".join(cmd))
    path = os.path.join(LOG_DIR, f"{SERVER}_server.log")
    fh = open(path, "a")
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            start_new_session=True, env=env)
    return proc, path


def kill_server(proc=None):
    """Kill the server's whole process tree (vLLM renames its workers VLLM::Worker_TP*), then wait for GPU memory."""
    if proc is not None:
        try:
            os.killpg(proc.pid, 9)
        except Exception:
            pass
    sh("pkill -9 -f '[v]llm.entrypoints' ; pkill -9 -f '[V]LLM::' ; pkill -9 -f '[s]glang.launch_server' ; "
       "pkill -9 -f '[s]glang::' ; sleep 5", timeout=60)
    for _ in range(24):
        _, out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits", timeout=30)
        used = [int(x) for x in out.split() if x.strip().isdigit()]
        if used and max(used) < 2000:
            break
        time.sleep(5)


def server_log_tail(path, n=60):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception:
        return ""


def wait_ready(proc, path, timeout):
    import urllib.request
    t = time.time()
    while time.time() - t < timeout:
        if proc.poll() is not None:
            log(f"server exited rc={proc.returncode}\n" + server_log_tail(path))
            return False
        try:
            with urllib.request.urlopen(BASE + "/v1/models", timeout=5) as r:
                if r.status == 200:
                    log(f"server ready in {time.time() - t:.0f}s")
                    return True
        except Exception:
            pass
        time.sleep(5)
    log("server not ready in time\n" + server_log_tail(path))
    return False


def server_capacity(path):
    """(kv_tokens, max_running) from the server log."""
    txt = open(path, errors="ignore").read()
    if SERVER == "sglang":
        kv, mr = None, None
        try:  # runtime values (after the Mamba cap); the log is the fallback
            import urllib.request
            with urllib.request.urlopen(BASE + "/get_server_info", timeout=20) as r:
                st = json.loads(r.read())["internal_states"][0]
            kv = st.get("memory_usage", {}).get("token_capacity")
            mr = st.get("effective_max_running_requests_per_dp")
        except Exception as exc:
            log("get_server_info failed:", exc)
        kvl = re.findall(r"KV Cache is allocated\. dtype: \S+ #tokens: (\d+)", txt)
        mrl = re.findall(r"max_running_requests is capped to (\d+)", txt)
        out = (kv or (int(kvl[-1]) if kvl else None), mr or (int(mrl[-1]) if mrl else None))
    else:
        m = re.findall(r"GPU KV cache size: ([\d,]+) tokens", txt)
        out = (int(m[-1].replace(",", "")) if m else None, None)
    log("server capacity (kv tokens, max running):", out)
    return out


_VLLM_METRICS = {"vllm:prefix_cache_queries_total": "pc_queries", "vllm:prefix_cache_hits_total": "pc_hits",
                 "vllm:num_preemptions_total": "preemptions", "vllm:kv_cache_usage_perc": "kv_usage",
                 "vllm:num_requests_running": "running", "vllm:num_requests_waiting": "waiting",
                 "vllm:prompt_tokens_total": "prompt_tokens", "vllm:generation_tokens_total": "gen_tokens"}


def server_stats(timeout=5.0):
    """Cache / load counters from the server (None if unavailable). Cumulative counters are summed over engines.

    vLLM: /metrics (prefix-cache queries/hits in tokens, preemptions, KV usage fraction, running/waiting).
    SGLang: /v1/loads (running/waiting, token_usage, cumulative uncached prefill tokens, retracted queue)."""
    import urllib.request
    try:
        if SERVER == "sglang":
            with urllib.request.urlopen(BASE + "/v1/loads", timeout=timeout) as r:
                ld = json.loads(r.read())["loads"]
            return {"running": sum(x.get("num_running_reqs", 0) for x in ld),
                    "waiting": sum(x.get("num_waiting_reqs", 0) for x in ld),
                    "kv_usage": max(x.get("token_usage", 0.0) for x in ld),
                    "kv_used_tokens": sum(x.get("num_used_tokens", 0) for x in ld),
                    "uncached_prefill": sum(x.get("total_prefill_uncached_tokens", 0) for x in ld),
                    "retracted": sum((x.get("queues") or {}).get("retracted", 0) for x in ld)}
        with urllib.request.urlopen(BASE + "/metrics", timeout=timeout) as r:
            txt = r.read().decode(errors="ignore")
        out = {}
        for line in txt.splitlines():
            if line.startswith("#"):
                continue
            name = line.split("{", 1)[0].split(" ", 1)[0]
            if name in _VLLM_METRICS:
                try:
                    v = float(line.rsplit(" ", 1)[1])
                except ValueError:
                    continue
                k = _VLLM_METRICS[name]
                out[k] = max(out.get(k, 0.0), v) if k == "kv_usage" else out.get(k, 0.0) + v
        return out or None
    except Exception:
        return None


def cache_line(cur, prev):
    """One progress-line fragment: prefix-cache hit rate over the window, KV usage, preemptions/retractions."""
    if not cur:
        return "server stats n/a"
    prev = prev or {}
    if SERVER == "sglang":
        new = cur["uncached_prefill"] - prev.get("uncached_prefill", 0)
        sent = cur.get("_client_prompt", 0) - prev.get("_client_prompt", 0)
        hit = f"{100 * (1 - new / sent):.1f}%" if sent > 0 else "n/a"
        return (f"cache hit {hit} (uncached prefill {new} of {sent}), running {cur['running']}, "
                f"waiting {cur['waiting']}, "
                f"KV usage {100 * cur['kv_usage']:.1f}%, retracted queue {cur['retracted']}")
    dq = cur.get("pc_queries", 0) - prev.get("pc_queries", 0)
    dh = cur.get("pc_hits", 0) - prev.get("pc_hits", 0)
    hit = f"{100 * dh / dq:.1f}%" if dq > 0 else "n/a"
    return (f"cache hit {hit} (of {dq:.0f} prompt tokens), running {cur.get('running', 0):.0f}, "
            f"waiting {cur.get('waiting', 0):.0f}, KV usage {100 * cur.get('kv_usage', 0):.1f}%, "
            f"preemptions {cur.get('preemptions', 0) - prev.get('preemptions', 0):.0f} "
            f"(total {cur.get('preemptions', 0):.0f})")


# ---------------------------------------------------------------- client

class Client:
    def __init__(self):
        import httpx
        self.last_prompt_tokens = None
        self.prompt_tokens = 0  # summed over finished requests (final usage), for the cache hit rate
        self.cached_tokens = 0  # usage.prompt_tokens_details.cached_tokens when the server reports it
        self.http = httpx.AsyncClient(base_url=BASE, timeout=httpx.Timeout(connect=30.0, read=900.0, write=120.0, pool=None),
                                      limits=httpx.Limits(max_connections=512, max_keepalive_connections=512))

    async def count(self, prompt):
        try:
            r = await self.http.post("/tokenize", json={"model": SERVED, "prompt": prompt})
            return int(r.json()["count"])
        except Exception:
            return int(len(prompt) / 2.5)

    async def stream(self, prompt, max_tokens, deadline, on_tokens, stop=None, info=None):
        """Streamed completion bounded by `deadline` even when no data arrives (finish 'deadline')."""
        parts = {"text": [], "ntok": 0}
        try:
            return await asyncio.wait_for(self._stream(prompt, max_tokens, deadline, on_tokens, stop, info, parts),
                                          timeout=max(1.0, deadline - time.time() + 5))
        except asyncio.TimeoutError:
            return "".join(parts["text"]), parts["ntok"], "deadline"

    async def _stream(self, prompt, max_tokens, deadline, on_tokens, stop, info, parts):
        """Streamed completion. Returns (text, completion_tokens, finish_reason); finish 'deadline' if cut off.

        Breaking out of the stream closes the connection, which makes vLLM abort the request.
        """
        body = {"model": SERVED, "prompt": prompt, "max_tokens": max_tokens, "temperature": TEMPERATURE,
                "top_p": 0.95, "top_k": 20, "stream": True, "skip_special_tokens": False,
                "stream_options": {"include_usage": True, "continuous_usage_stats": True}}
        if stop:
            body["stop"] = stop
        text, ntok, finish = parts["text"], 0, None
        last_usage = None
        async with self.http.stream("POST", "/v1/completions", json=body) as r:
            if r.status_code != 200:
                raise RuntimeError(f"HTTP {r.status_code}: {(await r.aread())[:300]!r}")
            async for line in r.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[6:]
                if data.strip() == "[DONE]":
                    break
                d = json.loads(data)
                if d.get("error"):
                    raise RuntimeError(f"server error event: {str(d['error'])[:300]}")
                u = d.get("usage")
                got_text = False
                for c in d.get("choices") or []:
                    text.append(c.get("text") or "")
                    got_text = got_text or bool(c.get("text"))
                    if c.get("finish_reason"):
                        finish = c["finish_reason"]
                if u:
                    last_usage = u
                if u and u.get("completion_tokens") is not None:
                    self.last_prompt_tokens = u.get("prompt_tokens") or self.last_prompt_tokens
                    if info is not None and u.get("prompt_tokens"):
                        info["prompt_tokens"] = u["prompt_tokens"]
                    on_tokens(u["completion_tokens"] - ntok)
                    ntok = u["completion_tokens"]
                elif got_text:  # server without continuous usage stats: ~1 token per chunk until usage arrives
                    on_tokens(1)
                    ntok += 1
                parts["ntok"] = ntok
                if time.time() > deadline:
                    finish = "deadline"
                    break
        if last_usage and last_usage.get("prompt_tokens"):
            self.prompt_tokens += last_usage["prompt_tokens"]
            self.cached_tokens += (last_usage.get("prompt_tokens_details") or {}).get("cached_tokens") or 0
        return "".join(text), ntok, finish or "stop"


# ---------------------------------------------------------------- tool executor (Kaggle hardening)

def robust_executor_factory(timeout):
    """E012 LocalExecutor (qwen-tools-3 executor: RLIMIT_AS/CPU/FSIZE, nice, oom_score_adj=1000, SIGINT to the
    process group, retried start with a longer timeout). nice 15 so tool code never starves the model server."""
    from arc_harness import executor as EX
    try:
        return EX.local_factory(timeout=timeout, nice=15)
    except TypeError:  # older executor without the hardening knobs
        return EX.local_factory(timeout=timeout)


# ---------------------------------------------------------------- scheduler

class Scheduler:
    def __init__(self, tasks, sub, client, force_at, hard_stop, concurrency, attempts=2, tag="run"):
        self.tasks, self.sub, self.client = tasks, sub, client
        self.force_at, self.hard_stop = force_at, hard_stop
        self.concurrency, self.attempts = concurrency, attempts
        self.pending1 = []
        self.pending2 = []
        self.inflight = {}  # job -> [budget, used]
        self.tokens = 0
        self.rate_samples = []  # (time, total tokens)
        self.gen_start = None
        self.records = []
        self.rec_path = os.path.join(LOG_DIR, f"attempts_{tag}.jsonl")
        self.launch_margin = 120  # no new attempts in the last 2 min before forcing
        self.retries = {}
        self.server_down = lambda: False  # set by main(): True while the vLLM server is dead or restarting
        self.cache_stats = []  # periodic server cache/load counters (summary)

    def on_tokens(self, n):
        self.tokens += n
        if "first_token" not in PHASES and n > 0:
            PHASES["first_token"] = round(time.time() - T0)

    def rate(self):
        now = time.time()
        self.rate_samples.append((now, self.tokens))
        self.rate_samples = [s for s in self.rate_samples if now - s[0] < 1800] or [(now, self.tokens)]
        t_old, k_old = self.rate_samples[0]
        if self.gen_start is None or now - self.gen_start < 900 or now - t_old < 300:
            return RATE_PRIOR
        return max(50.0, (self.tokens - k_old) / (now - t_old))

    def capacity_left(self):
        cap = self.rate() * max(0.0, self.force_at - time.time())
        return cap - sum(max(0, b - u) for b, u in self.inflight.values())

    def next_job(self):
        """(job, budget) or None. Attempt-1 jobs first; attempt 2 only with spare capacity."""
        if self.pending1:
            job = self.pending1.pop(0)
            share = self.capacity_left() / (len(self.pending1) + 1)
            return job, int(min(B_MAX1, max(B_MIN1, share)))
        while self.pending2:
            job = self.pending2.pop(0)
            share = self.capacity_left() / (len(self.pending2) + 1)
            if share < B_MIN2:
                continue
            return job, int(min(B_MAX2, share))
        return None

    async def attempt(self, job, budget):
        if AGENT == "tools":
            return await self.attempt_tools(job, budget)
        return await self.attempt_plain(job, budget)

    async def attempt_tools(self, job, budget):
        """One E012 tool-agent attempt (agent_qwen.solve + LocalExecutor) with deadline forcing.

        DeadlineLLM streams each completion; past force_at it cuts the stream and reports the attempt's budget as
        used, so the agent goes straight to its forced submit_answer (then the session `answer`). The forced call
        (stop=["</parameter>"]) may run until the hard stop."""
        import dataclasses
        from arc_harness import agent_qwen as AQ
        tid, p, a = job
        task = self.tasks[tid]
        cap = int(min(TOOLS_CAP, budget))
        extra = {"context": MAX_MODEL_LEN} if "context" in AQ.AgentConfig.__dataclass_fields__ else {}
        cfg = AQ.AgentConfig(**extra, tier=TOOLS_TIER, max_total_output=cap, budget_soft=min(TOOLS_SOFT, cap),
                             budget_notes=0.01, budget_warn=0.75, verify_submit=False, max_turns=60,
                             chunk_tokens=16384, force_submit_tokens=256, tool_timeout=TOOL_TIMEOUT,
                             temperature=TEMPERATURE, top_p=0.95, top_k=20)
        sch, client = self, self.client
        stats = {"tokens": 0, "requests": 0, "deadline_cuts": 0, "stream_errors": 0}

        class DeadlineLLM:
            async def complete(self, prompt, max_tokens, stop=None):
                info = {}
                stats["requests"] += 1
                if stop is None and time.time() >= sch.force_at:
                    stats["deadline_cuts"] += 1
                    return AQ.Completion("", cap, 0, "length")
                # forced submit expression: must end well before the hard-stop cancel so that submit_answer and
                # the session-answer fallback still run (review S2)
                dl = (max(sch.hard_stop - 180, min(time.time() + 30, sch.hard_stop - 10)) if stop
                      else sch.force_at)

                def cb(n):
                    stats["tokens"] += n
                    sch.on_tokens(n)
                    if job in sch.inflight:
                        sch.inflight[job][1] = stats["tokens"]
                try:
                    text, ntok, finish = await client.stream(prompt, max_tokens, dl, cb, stop=stop, info=info)
                except Exception as exc:
                    stats["stream_errors"] += 1
                    stats["last_stream_error"] = f"{type(exc).__name__}: {exc}"[:300]
                    if sch.server_down():
                        raise  # attempt is requeued once the server is back
                    # otherwise: end the attempt through the forced submit / session answer path
                    return AQ.Completion("", cap, 0, "length")
                if finish == "deadline":
                    stats["deadline_cuts"] += 1
                    return AQ.Completion(text, ntok + cap, info.get("prompt_tokens", 0), "length")
                return AQ.Completion(text, ntok, info.get("prompt_tokens", 0),
                                     "length" if finish == "length" else "stop")

        self.inflight[job] = [cap, 0]
        rec = {"task_id": tid, "pair": p, "attempt": a, "budget": cap, "agent": "tools"}
        t0 = time.time()
        try:
            transcript, res = await AQ.solve(DeadlineLLM(), robust_executor_factory(TOOL_TIMEOUT), task["train"],
                                             task["test"][p]["input"], cfg)
        except Exception as exc:
            transcript, res = "", {"pred": None, "error": f"{type(exc).__name__}: {exc}"[:400]}
        finally:
            self.inflight.pop(job, None)
        pred = res.get("pred")
        if hasattr(pred, "tolist"):
            pred = pred.tolist()
        rec.update(tokens=stats["tokens"], requests=stats["requests"], deadline_cuts=stats["deadline_cuts"],
                   stream_errors=stats["stream_errors"], last_stream_error=stats.get("last_stream_error"),
                   answered=pred is not None and valid_grid(pred), pred=pred,
                   forced=res.get("answer_source") in ("forced", "session") or stats["deadline_cuts"] > 0,
                   force_reason=("deadline" if stats["deadline_cuts"] else res.get("stop_reason")),
                   elapsed_s=round(time.time() - t0),
                   **{k: res.get(k) for k in ("answer_source", "stop_reason", "turns", "tool_calls", "tool_errors",
                                              "parse_failures", "eos_continues", "chunks", "seconds", "turn_log")})
        if res.get("error"):
            rec["error"] = res["error"]
        try:
            with open(os.path.join(LOG_DIR, f"trace_{tid}_{p}_{a}.txt"), "w") as f:
                f.write(transcript)
        except Exception:
            pass
        return rec

    async def attempt_plain(self, job, budget):
        tid, p, a = job
        task = self.tasks[tid]
        raw = render_prompt(build_prompt(task["train"], task["test"][p]["input"]))
        ptok = await self.client.count(raw)
        rec = {"task_id": tid, "pair": p, "attempt": a, "budget": budget, "prompt_tokens": ptok}
        self.inflight[job] = [budget, 0]
        gen, used, conts, streak, forced, force_reason, after_eos, chunks = "", 0, 0, 0, False, None, False, []
        t0 = time.time()
        try:
            while True:
                room = MAX_MODEL_LEN - ptok - used - 64
                if forced:
                    mt, dl = min(FINAL_TOKENS, room), self.hard_stop
                else:
                    mt, dl = min(budget - used, room - FINAL_TOKENS), self.force_at
                if mt <= 0 and not forced:
                    forced, force_reason = True, "budget"
                    gen += FORCE_TEXT
                    continue
                if mt <= 0:
                    break
                text, ntok, finish = await self.client.stream(raw + gen, mt, dl, self.on_tokens)
                gen += text
                used += ntok
                self.inflight[job][1] = used
                chunks.append([ntok, finish])
                if after_eos:
                    streak = streak + 1 if ntok < LOW_YIELD else 0
                if forced:
                    break
                if answer_section(gen) is not None:
                    if finish == "length" and used < budget and room - ntok > 64:
                        continue
                    break
                if finish == "deadline" or used >= budget or finish == "length":
                    forced, force_reason = True, ("deadline" if finish == "deadline" else "budget")
                    gen += FORCE_TEXT
                    continue
                # EOS mid-thought: continue, force after two low-yield continuations in a row
                if streak >= 2:
                    forced, force_reason = True, "low_yield_eos"
                    gen += FORCE_TEXT
                    continue
                conts += 1
                after_eos = True
        except Exception as exc:
            rec["error"] = f"{type(exc).__name__}: {exc}"[:400]
        finally:
            self.inflight.pop(job, None)
        content = answer_section(gen)
        pred = parse(content) if content is not None else None
        rec.update(tokens=used, continuations=conts, forced=forced, force_reason=force_reason, chunks=chunks,
                   elapsed_s=round(time.time() - t0), answered=pred is not None and valid_grid(pred), pred=pred,
                   tail=(content or gen[-300:])[-300:])
        with open(os.path.join(LOG_DIR, f"trace_{tid}_{p}_{a}.txt"), "w") as f:
            f.write(gen)
        return rec

    async def worker(self, wid):
        while time.time() < self.force_at - self.launch_margin:
            nj = self.next_job()
            if nj is None:
                return
            job, budget = nj
            rec = await self.attempt(job, budget)
            if "error" in rec and not rec["answered"] and self.server_down():
                # server crashed: requeue the job (at most twice) and wait for the watchdog to restart it
                self.retries[job] = self.retries.get(job, 0) + 1
                if self.retries[job] <= 2:
                    (self.pending1 if job[2] == 0 else self.pending2).insert(0, job)
                    log(f"server down; requeued {job}, waiting")
                    while self.server_down() and time.time() < self.force_at - self.launch_margin:
                        await asyncio.sleep(15)
                    continue
            self.records.append(rec)
            with open(self.rec_path, "a") as f:
                f.write(json.dumps(rec) + "\n")
            tid, p, a = job
            if rec["answered"]:
                self.sub.answers.setdefault((tid, p), {})[a] = rec["pred"]
                self.sub.write()
            if a == 0 and self.attempts > 1:
                self.pending2.append((tid, p, 1))
            log(f"done {tid}.{p}#{a} answered={rec['answered']} tok={rec['tokens']} budget={budget} "
                f"forced={rec['forced']}/{rec['force_reason']} "
                f"{rec['elapsed_s']}s {rec.get('error', '')}")

    async def monitor(self):
        last_t, last_k = time.time(), 0
        prev = None
        while True:
            await asyncio.sleep(120)
            now = time.time()
            r = (self.tokens - last_k) / (now - last_t)
            last_t, last_k = now, self.tokens
            cur = await asyncio.to_thread(server_stats)
            if cur is not None:
                cur["_client_prompt"] = getattr(self.client, "prompt_tokens", 0)
            log(f"progress: {len(self.records)} attempts done, in flight {len(self.inflight)}, pending1 "
                f"{len(self.pending1)}, pending2 {len(self.pending2)}, tokens {self.tokens}, window {r:.0f} tok/s, "
                f"rate est {self.rate():.0f} tok/s, avg {self.tokens / max(1, now - self.gen_start):.0f} tok/s | "
                + cache_line(cur, prev))
            if cur is not None:
                self.cache_stats.append({"t": round(now - T0), **cur})
                prev = cur

    async def run(self, jobs1):
        self.pending1 = list(jobs1)
        self.gen_start = time.time()
        mon = asyncio.create_task(self.monitor())
        try:
            await asyncio.wait_for(asyncio.gather(*(self.worker(i) for i in range(self.concurrency))),
                                   timeout=max(60, self.hard_stop - time.time()))
        except asyncio.TimeoutError:
            log("hard stop reached; cancelling in-flight attempts")
        mon.cancel()
        el = time.time() - self.gen_start
        log(f"scheduler finished: {len(self.records)} attempts, {self.tokens} tokens in {el / 60:.1f} min "
            f"({self.tokens / max(1, el):.0f} tok/s)")


def order_jobs(tasks):
    """Attempt-1 jobs, shortest prompts first (approximated by character length)."""
    jobs = [(tid, p, 0) for tid, t in tasks.items() for p in range(len(t["test"]))]
    size = {tid: len(json.dumps(t["train"])) + len(json.dumps(t["test"])) for tid, t in tasks.items()}
    return sorted(jobs, key=lambda j: (size[j[0]], j[0], j[1]))


# ---------------------------------------------------------------- benchmark (test runs only)

async def bench_decode(client, prefixes, n_seqs, gen_tokens, tasks, max_chars=None, n_unique=None):
    """Decode throughput at long context: real ARC prompts + real Qwen trace prefixes, n_seqs in parallel.

    n_unique: number of distinct prompts (sequence i uses prompt i % n_unique), so repeats can hit a prefix cache.
    """
    total = {"tok": 0}
    prompts = []
    for i in range(n_seqs):
        pr = prefixes[i % (n_unique or len(prefixes))]
        t = tasks[pr["task_id"]]
        prompts.append(render_prompt(build_prompt(t["train"], t["test"][pr["pair"]]["input"]))
                       + pr["text"][:max_chars])
    ctx = [await client.count(p) for p in prompts]
    log(f"bench: {n_seqs} seqs, context tokens mean {sum(ctx) / len(ctx):.0f} min {min(ctx)} max {max(ctx)}")
    first = {}
    t_start = time.time()

    def cb(n):
        total["tok"] += n

    async def one(i):
        def cbi(n):
            if i not in first:
                first[i] = time.time()
            cb(n)
        return await client.stream(prompts[i], gen_tokens, time.time() + 3600, cbi)

    samples = []

    async def sampler():
        while True:
            await asyncio.sleep(10)
            samples.append((time.time(), total["tok"]))
    s = asyncio.create_task(sampler())
    res = await asyncio.gather(*(one(i) for i in range(n_seqs)), return_exceptions=True)
    s.cancel()
    t_end = time.time()
    t_all_first = max(first.values()) if first else t_end
    dec = [x for x in samples if x[0] >= t_all_first]
    steady = (dec[-1][1] - dec[0][1]) / (dec[-1][0] - dec[0][0]) if len(dec) >= 2 else None
    errs = [repr(r)[:200] for r in res if isinstance(r, Exception)]
    out = {"n_seqs": n_seqs, "ctx_mean": sum(ctx) / len(ctx), "gen_tokens": gen_tokens, "total_tokens": total["tok"],
           "wall_s": t_end - t_start, "prefill_phase_s": t_all_first - t_start, "steady_decode_tok_s": steady,
           "overall_tok_s": total["tok"] / (t_end - t_start), "errors": errs}
    log("bench result:", json.dumps(out))
    return out


def score(sub, solutions, tasks):
    tot, pairs = 0.0, 0
    for tid, t in tasks.items():
        n = len(t["test"])
        ok = 0
        for p in range(n):
            s = sub[tid][p]
            ok += solutions[tid][p] in (s["attempt_1"], s["attempt_2"])
        tot += ok / n
        pairs += n
    return tot, len(tasks)


# ---------------------------------------------------------------- main

def setup_harness():
    """Make the bundled E012 harness importable and self-test the executor; False -> use the plain agent."""
    try:
        sys.path.insert(0, os.path.join(PKG, "evals"))
        from arc_harness import agent_qwen
        from arc_harness.executor import LocalExecutor, submit_answer
        agent_qwen.CONTEXT = MAX_MODEL_LEN  # the server's context, not the model's 262k
        t = time.time()
        with LocalExecutor([{"input": [[1, 2]], "output": [[2, 1]]}], [[3, 4]], timeout=3) as ex:
            r1 = ex.execute("import numpy as np\nx = np.array(test_input)[:, ::-1]\nprint(x.tolist())")
            r2 = ex.execute("print(x.sum())")
            r3 = ex.execute("1/0")
            r4 = ex.execute("while True:\n    pass")
            r5 = ex.execute("print(x.shape)")
            out, grid = submit_answer(ex, "x")
        ok = (r1["stdout"].strip() == "[[4, 3]]" and r2["stdout"].strip() == "7" and "ZeroDivisionError" in r3["stderr"]
              and r4["process_status"] == "timeout" and r5["stdout"].strip() == "(1, 2)" and grid == [[4, 3]])
        log(f"harness {agent_qwen.HARNESS_VERSION}; executor self-test {'OK' if ok else 'FAILED'} in "
            f"{time.time() - t:.1f}s")
        if not ok:
            log("self-test results:", json.dumps([r1, r2, r3, r4, r5, out])[:1500])
        return ok
    except Exception as exc:
        log(f"harness/executor self-test error: {type(exc).__name__}: {exc}")
        return False


def is_rerun(test_path):
    """Belt and braces: the env var, or a test file whose task ids differ from the public one."""
    import hashlib
    env = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
    try:
        ids = sorted(json.load(open(test_path)))
        differs = hashlib.sha256(",".join(ids).encode()).hexdigest() != PUBLIC_TEST_IDS_SHA256
    except Exception:
        differs = False
    return env or differs, env, differs


def emergency_timer():
    """Last-resort timer (review S1): at T0 + ARC_EMERGENCY_S, make sure a valid submission.json exists, kill the
    server and all children, and exit, whatever the asyncio side is doing."""
    import threading

    def fire():
        log("EMERGENCY timer fired: validating submission.json, killing children, exiting")
        try:
            path = os.path.join(WORK, "submission.json")
            tasks = json.load(open(comp_file("arc-agi_test_challenges.json")))
            try:
                check_submission(path, tasks)
            except Exception:
                global OUT_SUB
                OUT_SUB = path
                Submission(tasks).write()
        finally:
            subprocess.run("pkill -9 -f '[V]LLM::'; pkill -9 -f '[v]llm.entrypoints'; pkill -9 -f '[s]glang'",
                           shell=True)
            os._exit(0)
    t = threading.Timer(max(1.0, T0 + envf("ARC_EMERGENCY_S", 11.7 * 3600) - time.time()), fire)
    t.daemon = True
    t.start()


def main():
    global OUT_SUB
    emergency_timer()
    os.makedirs(LOG_DIR, exist_ok=True)
    driver_env()
    global AGENT
    rerun, env_rerun, ids_differ = is_rerun(comp_file("arc-agi_test_challenges.json"))
    mode = "submit" if rerun else MODE
    log(f"rerun={rerun} (env {env_rerun}, test ids differ from public {ids_differ})")
    log(f"mode={mode} server={SERVER} agent={AGENT} rerun={rerun} "
        f"T0={time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(T0))} UTC")
    test_tasks = json.load(open(comp_file("arc-agi_test_challenges.json")))
    Submission(test_tasks).write()  # valid fallback for the competition test file, always
    log(f"fallback submission written ({len(test_tasks)} tasks)")
    if mode == "fallback":
        return
    if mode == "local":
        # Non-rerun commit/interactive run: small local test set with known solutions, self-scored, time-boxed.
        tasks = json.load(open(find_one(f"{INPUT}/**/arc-agi_local_challenges.json")))
        solutions = json.load(open(find_one(f"{INPUT}/**/arc-agi_local_solutions.json")))
        if LOCAL_TASKS:
            tasks = {t: tasks[t] for t in LOCAL_TASKS.split(",") if t in tasks}
        force_at = hard_stop = None  # set once the server is ready
        OUT_SUB = os.path.join(WORK, "local_submission.json")
    else:
        tasks, solutions = test_tasks, None
        force_at, hard_stop = T0 + FORCE_AT_S, T0 + HARD_STOP_S
    sub = Submission(tasks)
    sub.write()
    log(f"{mode}: {len(tasks)} tasks, {sum(len(t['test']) for t in tasks.values())} test inputs"
        + (f"; force at {(force_at - T0) / 3600:.2f}h, hard stop {(hard_stop - T0) / 3600:.2f}h" if force_at else ""))
    if AGENT == "tools" and not setup_harness():
        AGENT = "plain"
        log("falling back to the plain (no-tool) agent")
    PHASES["harness_ready"] = round(time.time() - T0)

    if EXTERNAL:
        # off-Kaggle validation against an already running server: no install, no launch, no watchdog restarts
        class _Alive:
            returncode = None

            def poll(self):
                return None
        mdir, proc, slog = None, _Alive(), os.path.join(LOG_DIR, "external_server.log")
        open(slog, "a").close()
        log(f"external server {BASE} (model {SERVED}); skipping install and launch")
    else:
        mdir = model_dir()
        log("model dir:", mdir)
        install_server()
        PHASES["installed"] = round(time.time() - T0)
        preload(mdir)
        PHASES["preloaded"] = round(time.time() - T0)
        proc, slog = launch_server(mdir)
    if not wait_ready(proc, slog, timeout=1800):
        log("server start failed; killing the process tree and retrying once")
        kill_server(proc)
        proc, slog = launch_server(mdir)
        if not wait_ready(proc, slog, timeout=1800):
            raise RuntimeError("server failed to start (twice)")
    PHASES["server_ready"] = round(time.time() - T0)
    if mode == "local":
        if LOCAL_READY_MIN > 0:
            force_at = time.time() + LOCAL_READY_MIN * 60
        else:
            force_at = min(T0 + LOCAL_FORCE_CAP_MIN * 60, max(T0 + LOCAL_FORCE_MIN * 60, time.time() + 20 * 60))
        hard_stop = force_at + LOCAL_HARD_MIN * 60
        log(f"local time box: force at {(force_at - T0) / 60:.0f} min, hard stop {(hard_stop - T0) / 60:.0f} min")
    kv, max_running = server_capacity(slog)
    conc = CONCURRENCY or min(MAX_SEQS, max_running or MAX_SEQS,
                              max(8, int(kv / TOKENS_PER_SEQ_KV)) if kv else 16)
    log(f"concurrency {conc} (max_seqs {MAX_SEQS}, server max running {max_running}, kv {kv})")
    server = {"proc": proc, "down": False, "restarts": 0}

    async def watchdog(force_at):
        """Restart the server (at most twice) if it dies while there is time left to use it."""
        while True:
            await asyncio.sleep(15)
            if server["proc"].poll() is None:
                continue
            server["down"] = True
            log(f"server died (rc={server['proc'].returncode}):\n" + server_log_tail(slog, 40))
            if server["restarts"] >= 2 or force_at - time.time() < 45 * 60:
                log("not restarting the server")
                return
            server["restarts"] += 1
            kill_server(server["proc"])
            server["proc"], _ = launch_server(mdir)
            ok = await asyncio.to_thread(wait_ready, server["proc"], slog, 1800)
            log(f"server restart {'ok' if ok else 'FAILED'}")
            if not ok:
                return
            server["down"] = False

    async def cpu_monitor():
        while True:
            await asyncio.sleep(120)
            try:
                la = os.getloadavg()
                log(f"cpu load avg {la[0]:.1f}/{la[1]:.1f}/{la[2]:.1f} on {os.cpu_count()} cpus")
            except Exception:
                pass

    async def go():
        from concurrent.futures import ThreadPoolExecutor
        # agent_qwen runs executor calls via asyncio.to_thread: one thread per concurrent attempt (E012)
        asyncio.get_running_loop().set_default_executor(ThreadPoolExecutor(max_workers=conc + 8))
        client = Client()
        sch = Scheduler(tasks, sub, client, force_at, hard_stop, conc, attempts=2, tag=mode)
        sch.server_down = lambda: server["down"] or server["proc"].poll() is not None
        wd = asyncio.create_task(watchdog(sch.force_at))
        cm = asyncio.create_task(cpu_monitor())
        await sch.run(order_jobs(tasks))
        wd.cancel()
        cm.cancel()
        sub.write()
        check_submission(OUT_SUB, tasks)
        recs = sch.records
        summary = {"mode": mode, "server": SERVER, "agent": AGENT, "concurrency": conc, "kv_tokens": kv,
                   "phases_s": PHASES, "server_restarts": server["restarts"],
                   "attempts": len(recs), "tokens": sch.tokens,
                   "gen_minutes": round((time.time() - sch.gen_start) / 60, 1),
                   "answered": sum(r["answered"] for r in recs), "errors": sum("error" in r for r in recs),
                   "stream_errors": sum(r.get("stream_errors") or 0 for r in recs),
                   "server_preemption_lines": sum("preempt" in ln.lower() for ln in open(slog, errors="ignore")),
                   "cache_stats": sch.cache_stats[-1] if sch.cache_stats else None,
                   "cache_hit_overall": cache_line(sch.cache_stats[-1], None) if sch.cache_stats else None,
                   "kv_usage_max_pct": max([float(x) for x in re.findall(r"KV cache usage: ([\d.]+)%",
                                                                          open(slog, errors="ignore").read())] or [0]),
                   "tool_calls": sum(r.get("tool_calls") or 0 for r in recs),
                   "tool_errors": sum(r.get("tool_errors") or 0 for r in recs),
                   "parse_failures": sum(r.get("parse_failures") or 0 for r in recs),
                   "answer_sources": {k: sum(r.get("answer_source") == k for r in recs)
                                      for k in ("submit", "forced", "session", "salvage", "parse", "lenient", None)}}
        if solutions:
            sc, n = score(json.load(open(OUT_SUB)), solutions, tasks)
            summary["score"] = f"{sc:.2f}/{n} tasks"
            summary["per_pair"] = [{"task": r["task_id"], "pair": r["pair"], "attempt": r["attempt"],
                                    "correct": r["pred"] == solutions[r["task_id"]][r["pair"]],
                                    "tokens": r["tokens"], "turns": r.get("turns"), "tool_calls": r.get("tool_calls"),
                                    "tool_errors": r.get("tool_errors"), "source": r.get("answer_source"),
                                    "stop": r.get("stop_reason"), "s": r["elapsed_s"], "error": r.get("error")}
                                   for r in recs]
        json.dump(summary, open(os.path.join(WORK, f"summary_{mode}.json"), "w"), indent=1)
        log("SUMMARY", json.dumps(summary))
    asyncio.run(go())
    log("server log tail:\n" + server_log_tail(slog, 10))


if __name__ == "__main__":
    try:
        main()
    except BaseException as exc:  # keep whatever submission.json exists
        import traceback
        traceback.print_exc()
        log(f"FATAL {type(exc).__name__}: {exc}")
    finally:
        try:
            tasks_path = comp_file("arc-agi_test_challenges.json")
            check_submission(os.path.join(WORK, "submission.json"), json.load(open(tasks_path)))
            log("submission.json valid")
        except Exception as exc:
            log(f"submission check failed ({exc}); rewriting fallback")
            OUT_SUB = os.path.join(WORK, "submission.json")
            Submission(json.load(open(comp_file("arc-agi_test_challenges.json")))).write()
        os._exit(0)


In [ ]:
%%writefile /kaggle/working/arc_solver/replay_bench.py
"""E014: agentic replay benchmark on Kaggle 4xL4 (offline). Runs next to solver.py and uses its install / launch /
kill / stats code, so the bench exercises the same server path as the submission.

Workload (dataset chankhavu/arc2-agentic-replay, built by kaggle/datasets/agentic_replay/build.py from E012 Qwen
tool-use transcripts): every agent replays one recorded attempt turn by turn. Turn t sends prompt_t and asks for
exactly the recorded number of generated tokens (ignore_eos, temp 1.0 / top_p 0.95 / top_k 20, chunks of 16384 like
the agent); then prompt_{t+1} = prompt_t + <the server's own generated text> + <recorded tool output + next
assistant header>, which is what the real agent sends, so prefix caching sees the real reuse pattern. A finished
attempt is followed by the next transcript from turn 0. Agents that join at a level start are warm-started at a
random turn (stationary distribution truncated to contexts <= ARC_REPLAY_WARM_MAX, recorded history as a cold
prompt), so the live working set is closer to a long run than a cold start.

Plan (env ARC_REPLAY_PLAN, JSON list): [{"name", "env": {ARC_* settings for launch_server}, "max_seqs",
"levels": [[agents, seconds], ...], "fallback_envs": [{...}, ...], or "base" + "extra_env"}]. Concurrency ramps up within one server start; per level
we report decode tok/s, new-prefill tok/s, prefix-cache hit rate (client: cached/prompt tokens; server counters),
hit vs ideal on continuation turns (ideal = previous request's prompt + completion), running / waiting requests,
KV usage, preemptions (vLLM) / retractions (SGLang), TTFT and turn time. Results: /kaggle/working/replay_bench.json.
Time-boxed by ARC_BENCH_STOP_MIN (minutes from notebook start).
"""
import asyncio
import glob
import gzip
import json
import os
import random
import re
import shutil
import sys
import threading
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import solver as S  # noqa: E402

STOP = S.T0 + float(os.environ.get("ARC_BENCH_STOP_MIN", 60)) * 60
OUT = os.path.join(S.WORK, "replay_bench.json")
TOOL_DELAY = float(os.environ.get("ARC_REPLAY_TOOL_DELAY", 1.0))
WARM_MAX = int(os.environ.get("ARC_REPLAY_WARM_MAX", 20000))
CHUNK = 16384
SAMPLE_S = 10.0
RESULTS = {"configs": {}, "phases": {}}


def save():
    tmp = OUT + ".tmp"
    json.dump(RESULTS, open(tmp, "w"), indent=1)
    os.replace(tmp, OUT)


def left():
    return STOP - time.time()


def load_data():
    p = S.find_one(f"{S.INPUT}/**/agentic_replay.json.gz") or S.find_one(f"{S.INPUT}/**/agentic_replay.json")
    if not p:  # Kaggle stores uploaded .gz files decompressed
        raise FileNotFoundError("agentic_replay.json(.gz) not attached")
    return json.load(gzip.open(p, "rt") if p.endswith(".gz") else open(p))


def check_gpus():
    """Abort at once (saving GPU quota) unless the session really has 4 L4s: a push with machine_shape NvidiaL4
    once landed on a 2-GPU / 31 GB host."""
    _, out = S.sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", timeout=60)
    _, mem = S.sh("free -g | sed -n 2p", timeout=30)
    gpus = [ln.strip() for ln in out.splitlines() if ln.strip()]
    RESULTS["gpus"], RESULTS["host_mem"] = gpus, mem.strip()
    S.log("gpus:", gpus, "| host:", mem.strip(), "| cpus:", os.cpu_count())
    want = int(os.environ.get("ARC_REQUIRE_GPUS", 4))
    if len(gpus) != want or not all("L4" in g for g in gpus):
        raise RuntimeError(f"expected {want}x L4, got {gpus}")


def warm_start(x, rng):
    """(turn index, prompt) for an agent joining mid-run: turn drawn with weight = its generated tokens among turns
    whose start context <= WARM_MAX; the prompt is the recorded transcript up to that turn."""
    cand = [(i, t["gen"]) for i, t in enumerate(x["turns"]) if t["ctx"] <= WARM_MAX and t["gen"] > 0]
    if not cand:
        return 0, x["prompt0"]
    r, acc, k = rng.random() * sum(w for _, w in cand), 0.0, cand[-1][0]
    for i, w in cand:
        acc += w
        if acc >= r:
            k = i
            break
    return k, x["prompt0"] + "".join(x["hist"][i] + x["turns"][i]["suffix"] for i in range(k))


class Replay:
    def __init__(self, data, name):
        import httpx
        self.data, self.name = data, name
        self.http = httpx.AsyncClient(base_url=S.BASE, timeout=httpx.Timeout(connect=30.0, read=1500.0, write=120.0,
                                                                              pool=None),
                                      limits=httpx.Limits(max_connections=256, max_keepalive_connections=256))
        self.next_x = 0
        self.reqs = []        # one record per request
        self.samples = []     # periodic server + client state
        self.tokens = 0       # streamed completion tokens (all agents)
        self.live = {}        # agent -> current context tokens (prompt + generated so far)
        self.level = None
        self.errors = 0

    def take(self):
        x = self.data[self.next_x % len(self.data)]
        self.next_x += 1
        return x

    async def request(self, agent, prompt, max_tokens, ideal):
        body = {"model": S.SERVED, "prompt": prompt, "max_tokens": max_tokens, "temperature": 1.0, "top_p": 0.95,
                "top_k": 20, "ignore_eos": True, "stream": True, "skip_special_tokens": False,
                "stream_options": {"include_usage": True, "continuous_usage_stats": True}}
        rec = {"agent": agent, "level": self.level, "t0": time.time(), "ideal": ideal, "max_tokens": max_tokens}
        text, ntok, usage, base = [], 0, None, self.live.get(agent, 0)
        try:
            async with self.http.stream("POST", "/v1/completions", json=body) as r:
                if r.status_code != 200:
                    raise RuntimeError(f"HTTP {r.status_code}: {(await r.aread())[:300]!r}")
                async for line in r.aiter_lines():
                    if not line.startswith("data: "):
                        continue
                    d = line[6:]
                    if d.strip() == "[DONE]":
                        break
                    d = json.loads(d)
                    if d.get("error"):
                        raise RuntimeError(str(d["error"])[:300])
                    got = False
                    for c in d.get("choices") or []:
                        if c.get("text"):
                            text.append(c["text"])
                            got = True
                    if got and "t_first" not in rec:
                        rec["t_first"] = time.time()
                    u = d.get("usage")
                    if u:
                        usage = u
                    if u and u.get("completion_tokens") is not None:
                        self.tokens += u["completion_tokens"] - ntok
                        ntok = u["completion_tokens"]
                        if u.get("prompt_tokens"):
                            self.live[agent] = u["prompt_tokens"] + ntok
                    elif got:
                        self.tokens += 1
                        ntok += 1
                        self.live[agent] = base + ntok
        except asyncio.CancelledError:
            rec["cancelled"] = True
            raise
        except Exception as exc:
            rec["error"] = f"{type(exc).__name__}: {exc}"[:300]
            self.errors += 1
        finally:
            rec["t1"] = time.time()
            rec["completion"] = ntok
            if usage:
                rec["prompt"] = usage.get("prompt_tokens")
                rec["cached"] = (usage.get("prompt_tokens_details") or {}).get("cached_tokens")
            self.reqs.append(rec)
        return "".join(text), ntok, rec

    async def agent(self, idx, warm):
        rng = random.Random(1000 + idx)
        first = True
        while True:
            x = self.take()
            k, prompt = warm_start(x, rng) if (warm and first) else (0, x["prompt0"])
            first = False
            ideal, last_prompt = 0, 0
            for ti in range(k, len(x["turns"])):
                t = x["turns"][ti]
                remaining, text = t["gen"], ""
                while remaining > 0:
                    mt = min(CHUNK, remaining)
                    out, n, rec = await self.request(idx, prompt + text, mt, ideal)
                    if rec.get("error") or n <= 0:
                        await asyncio.sleep(5)
                        break
                    text += out
                    remaining -= n
                    last_prompt = rec.get("prompt") or last_prompt
                    ideal = last_prompt + n
                if ti == len(x["turns"]) - 1 or last_prompt + 2 * CHUNK > S.MAX_MODEL_LEN:
                    break
                await asyncio.sleep(TOOL_DELAY)  # tool execution
                prompt = prompt + text + t["suffix"]
            self.live[idx] = 0

    async def sampler(self):
        while True:
            st = await asyncio.to_thread(S.server_stats, 8.0)
            self.samples.append({"t": time.time(), "level": self.level, "tokens": self.tokens,
                                 "live_ctx": sum(self.live.values()), "agents_live": sum(v > 0 for v in self.live.values()),
                                 "server": st})
            await asyncio.sleep(SAMPLE_S)

    async def run(self, levels):
        tasks, n = [], 0
        smp = asyncio.create_task(self.sampler())
        marks = []
        try:
            for li, (agents, dur) in enumerate(levels):
                if left() < 90:
                    S.log(f"[{self.name}] time box: skipping level {agents}")
                    break
                dur = min(dur, left() - 60)
                self.level = agents
                t_start = time.time()
                while n < agents:
                    tasks.append(asyncio.create_task(self.agent(n, warm=True)))
                    n += 1
                S.log(f"[{self.name}] level {agents} agents for {dur:.0f}s")
                await asyncio.sleep(dur)
                marks.append((agents, t_start, time.time()))
                S.log(f"[{self.name}] level {agents} done: " + json.dumps(self.summarize(agents, t_start, time.time())))
        finally:
            for t in tasks:
                t.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            smp.cancel()
            await self.http.aclose()
        return marks

    def summarize(self, agents, t_start, t_end):
        skip = min(90.0, (t_end - t_start) / 3)
        w0 = t_start + skip
        span = max(1.0, t_end - w0)
        sm = [s for s in self.samples if w0 <= s["t"] <= t_end]
        out = {"agents": agents, "window_s": round(span)}
        if len(sm) >= 2:
            out["decode_tok_s"] = round((sm[-1]["tokens"] - sm[0]["tokens"]) / (sm[-1]["t"] - sm[0]["t"]), 1)
            out["live_ctx_mean"] = round(sum(s["live_ctx"] for s in sm) / len(sm))
            out["live_ctx_max"] = max(s["live_ctx"] for s in sm)
            srv = [s["server"] for s in sm if s["server"]]
            if srv:
                out["running_mean"] = round(sum(x.get("running", 0) for x in srv) / len(srv), 1)
                out["running_max"] = max(x.get("running", 0) for x in srv)
                out["waiting_max"] = max(x.get("waiting", 0) for x in srv)
                out["kv_usage_mean"] = round(sum(x.get("kv_usage", 0) for x in srv) / len(srv), 3)
                out["kv_usage_max"] = round(max(x.get("kv_usage", 0) for x in srv), 3)
                a, b = srv[0], srv[-1]
                if "pc_queries" in b:
                    dq = b["pc_queries"] - a.get("pc_queries", 0)
                    out["server_hit_rate"] = round((b["pc_hits"] - a.get("pc_hits", 0)) / dq, 4) if dq else None
                    out["preemptions"] = b.get("preemptions", 0) - a.get("preemptions", 0)
                    out["server_new_prefill_tok_s"] = round((dq - (b["pc_hits"] - a.get("pc_hits", 0))) / span, 1)
                if "uncached_prefill" in b:
                    out["server_new_prefill_tok_s"] = round((b["uncached_prefill"] - a["uncached_prefill"]) / span, 1)
                    out["retracted_queue_max"] = max(x.get("retracted", 0) for x in srv)
        rq = [r for r in self.reqs if w0 <= r["t0"] <= t_end and r.get("prompt")]
        if rq:
            pt = sum(r["prompt"] for r in rq)
            ct = sum(r.get("cached") or 0 for r in rq)
            out["requests"] = len(rq)
            out["client_hit_rate"] = round(ct / pt, 4) if any(r.get("cached") is not None for r in rq) else None
            out["client_new_prefill_tok_s"] = round((pt - ct) / span, 1)
            cont = [r for r in rq if r["ideal"] > 0 and r.get("cached") is not None]
            if cont:
                out["cont_hit_vs_ideal"] = round(sum(min(r["cached"], r["ideal"]) for r in cont)
                                                 / sum(r["ideal"] for r in cont), 4)
                out["cont_miss_frac"] = round(sum(r["cached"] < 0.9 * r["ideal"] for r in cont) / len(cont), 4)
            tt = sorted(r["t_first"] - r["t0"] for r in rq if r.get("t_first"))
            if tt:
                out["ttft_p50"] = round(tt[len(tt) // 2], 2)
                out["ttft_p90"] = round(tt[int(len(tt) * 0.9)], 2)
            dur = sorted(r["t1"] - r["t0"] for r in rq if not r.get("error"))
            if dur:
                out["req_s_p50"] = round(dur[len(dur) // 2], 1)
            out["errors"] = sum(1 for r in rq if r.get("error"))
        return out


def log_lines(path):
    keep = re.compile(r"Decode batch|Prefill batch|retract|evict|preempt|Engine 000|KV Cache|Mamba|capped|"
                      r"hicache|HiCache|host|error|Error", re.I)
    try:
        return [ln.rstrip()[:400] for ln in open(path, errors="ignore") if keep.search(ln)][-2500:]
    except Exception:
        return []


async def run_config(cfg, data, mdir):
    name = cfg["name"]
    res = RESULTS["configs"].setdefault(name, {"cfg": cfg})
    if cfg.get("base"):  # same server settings as the config that started successfully, plus extra_env
        base = RESULTS["configs"].get(cfg["base"], {}).get("ok_env")
        if base is None:
            res["skipped"] = f"base config {cfg['base']} did not start"
            save()
            return
        cfg = {**cfg, "env": {**base, **cfg.get("extra_env", {})}, "fallback_env": None, "fallback_envs": None}
    attempts = [cfg.get("env", {})] + ([cfg["fallback_env"]] if cfg.get("fallback_env") is not None else []) \
        + list(cfg.get("fallback_envs") or [])
    proc = path = None
    for ai, env in enumerate(attempts):
        if left() < 10 * 60:
            res["skipped"] = f"{left() / 60:.1f} min left"
            save()
            return
        for k in list(os.environ):
            if k.startswith(("ARC_SGL_", "ARC_SERVER_EXTRA")):
                del os.environ[k]
        os.environ.update({k: str(v) for k, v in env.items()})
        S.MAX_SEQS = int(cfg.get("max_seqs", S.MAX_SEQS))
        t = time.time()
        proc, path = S.launch_server(mdir)
        ok = await asyncio.to_thread(S.wait_ready, proc, path, min(1500, max(60, left() - 300)))
        res.setdefault("starts", []).append({"env": env, "ready": ok, "startup_s": round(time.time() - t)})
        if ok:
            res["ok_env"] = env
            break
        res["starts"][-1]["log_tail"] = S.server_log_tail(path, 60)
        S.kill_server(proc)
        dst = os.path.join(S.LOG_DIR, f"{name}_failed{ai}.log")
        shutil.move(path, dst)
        save()
    else:
        return
    kv, mr = S.server_capacity(path)
    res["capacity"] = {"kv_tokens": kv, "max_running": mr}
    res["server_stats_at_start"] = S.server_stats()
    save()
    rp = Replay(data, name)
    try:
        marks = await rp.run(cfg["levels"])
    except Exception as exc:
        res["replay_error"] = f"{type(exc).__name__}: {exc}"[:500]
        marks = []
    res["levels"] = [rp.summarize(a, t0, t1) for a, t0, t1 in marks]
    res["errors"] = rp.errors
    res["samples"] = [{**s, "t": round(s["t"] - S.T0, 1)} for s in rp.samples]
    res["requests"] = [{k: (round(v - S.T0, 2) if k in ("t0", "t1", "t_first") else v) for k, v in r.items()}
                       for r in rp.reqs]
    res["log_lines"] = log_lines(path)
    res["proc_alive_at_end"] = proc.poll() is None
    save()
    S.kill_server(proc)
    try:
        shutil.move(path, os.path.join(S.LOG_DIR, f"{name}_server.log"))
    except Exception:
        pass
    S.log(f"[{name}] levels: " + json.dumps(res["levels"]))


async def main_async(plan):
    data = load_data()
    RESULTS["n_transcripts"] = len(data)
    mdir = S.model_dir()
    for cfg in plan:
        await run_config(cfg, data, mdir)
        RESULTS["phases"][f"{cfg['name']}_done"] = round(time.time() - S.T0)
        save()


def main():
    os.makedirs(S.LOG_DIR, exist_ok=True)
    S.driver_env()
    plan = json.loads(os.environ["ARC_REPLAY_PLAN"])
    RESULTS["plan"] = plan
    S.log(f"replay bench start: server {S.SERVER}, stop at {(STOP - S.T0) / 60:.0f} min")
    # Hard time box independent of the event loop: final save + kill the server tree, then exit.
    def guard():
        while time.time() < STOP + 60:
            time.sleep(5)
        S.log("replay bench guard: time box exceeded; killing the server and exiting")
        try:
            RESULTS["guard_fired"] = True
            save()
        finally:
            S.kill_server()
            os._exit(0)
    threading.Thread(target=guard, daemon=True).start()
    try:
        check_gpus()
        load_data()  # fail before the install if the dataset is missing
        mdir = S.model_dir()
        pre = threading.Thread(target=S.preload, args=(mdir,), daemon=True)
        pre.start()  # page-cache the 31 GB model while the wheels install
        S.install_server()
        RESULTS["phases"]["installed"] = round(time.time() - S.T0)
        pre.join(timeout=max(1, left() - 600))
        RESULTS["phases"]["preloaded"] = round(time.time() - S.T0)
        save()
        asyncio.run(main_async(plan))
    except BaseException as exc:
        import traceback
        traceback.print_exc()
        RESULTS["fatal"] = f"{type(exc).__name__}: {exc}"[:500]
    finally:
        jit = os.environ.get("ARC_SGL_JIT_DIR", "/tmp/sglang_jit")
        if S.SERVER == "sglang" and os.path.isdir(jit):
            shutil.copytree(jit, os.path.join(S.WORK, "sglang_jit_cache_gpu"), symlinks=True, dirs_exist_ok=True)
        RESULTS["phases"]["end"] = round(time.time() - S.T0)
        save()
        S.kill_server()
        S.log("replay bench done: " + json.dumps({k: v.get("levels") for k, v in RESULTS["configs"].items()})[:6000])


if __name__ == "__main__":
    main()
    os._exit(0)


In [ ]:
import os, signal, subprocess, sys, threading, time
KILL_AT = float(os.environ['ARC_T0']) + 3720  # outer guard: time box + 4 min
p = subprocess.Popen([sys.executable, '-u', '/kaggle/working/arc_solver/replay_bench.py'], stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, cwd='/kaggle/working', start_new_session=True)
def guard():
    while p.poll() is None and time.time() < KILL_AT:
        time.sleep(10)
    if p.poll() is None:
        print('OUTER GUARD: killing the bench process group', flush=True)
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except Exception:
            pass
        os.system("pkill -9 -f '[s]glang' ; pkill -9 -f '[v]llm' ; pkill -9 -f '[V]LLM::'")
threading.Thread(target=guard, daemon=True).start()
for line in p.stdout:
    print(line, end='', flush=True)
print('bench exit code', p.wait())

In [ ]:
import json
r = json.load(open('/kaggle/working/replay_bench.json'))
print(json.dumps({k: {'starts': v.get('starts'), 'capacity': v.get('capacity'), 'levels': v.get('levels')}
                  for k, v in r['configs'].items()}, indent=1)[:20000])
print(r.get('phases'), r.get('fatal'))